# Demo: Controlled Pendulum
## Analysis of Different Co-Simulation Algorithms

-------------------------
**References:**

Gomes, C., Thule, C., Broman, D., Larsen, P. G., & Vangheluwe, H.  
**Co-Simulation: A Survey.** *ACM Computing Surveys (CSUR)*, 51(3), Article 49, 2018.  
https://doi.org/10.1145/3179993

Petridis, K., & Clauß, C. (2015). **Test of Basic Co-Simulation Algorithms Using FMI**. Robert Bosch GmbH & Fraunhofer IIS EAS. 
https://2015.international.conference.modelica.org/proceedings/html/submissions/ecp15118865_PetridisClau.pdf

S. Sicklinger, V. Belsky, B. Engelmann, H. Elmqvist, H. Olsson, R. Wüchner, and K.-U. Bletzinger.  
**Interface Jacobian-based Co-Simulation.**  
*International Journal for Numerical Methods in Engineering*, 98(4):418–444, 2014.  
doi:10.1002/nme.4637.
https://onlinelibrary.wiley.com/doi/abs/10.1002/nme.4637

-------------------------

### **1. Introduction Co-Simulation**

- **Core Concept:** Simulate a global system by composing multiple independent subsystems, each potentially using different simulation tools or solvers.
    - Computing the behavior of the combined models over time.
    - Each simulation unit with its own interface for getting, setting inputs/outputs and computing the behavior of its model over a given interval of time.
- **Tool Incompatibility:** Models from different tools are difficult to exchange and integrate
- **Interface Coupling:** Components exchange data through well-defined input/output interfaces
- **Master algorithm:** scheduling execution and communication of each co-simulation unit
    - representing a system assembled from models in various domains; each with their own appropriate simulator
	- black-box-models which hide internal details
- **Problem:** difficult to ensure that the results produced by a co-simulation can be trusted
    - communication frequency betweeen different differential-equation-based units
	- event propagation order
	- numerical properties of the participating units
	- users do not always know how to configure the co-simulation
- **Problem of algebraic loops:** Arise when differential-algebraic equation-based units are coupled
	- Solve algebraic loops by fixed point iteration
	- Requires simulation units support state rollback
	- OpenModelica exported FMUs do not support state rollback directly

### **2. Basics on Co-Simulation Algorithms**

#### **2.1 Dynamical systems and behavior**

- A **dynamical system** is an abstract model of a real system (physical or computational) characterized by:
	- state: $x(t)$  
	- inputs: $u(t)$  
	- outputs: $y(t)$  
	- evolution rules (typically ODEs/DAEs)

- **State–space representation:**
	- Continuous-time form (general):
		$$\dot{x} = f(x, u, t),\qquad y = g(x, u, t)$$
	- Where:
		- $\dot{x}$ denotes the time derivative of the state,
		- $f$ describes state evolution,
		- $g$ maps state and inputs to outputs.

- **Behavior:**
	- The behavior of the system is the set of trajectories $(x(t), y(t))$ that satisfy the evolution equations under a given experimental frame (assumptions, boundary conditions, input signals, etc.).
	- Simulation time $t$ is the independent time variable; it may run faster, slower, or equal to wall‑clock time depending on the simulator.

- **Simulator (solver):**
	- A simulator is a numerical algorithm that approximates trajectories $(x(t), y(t))$.
	- Accuracy depends on:
		- numerical method (e.g., explicit/implicit integrators),
		- step size and step control,
		- how continuous signals are represented and discretized (interpolation, sample-and-hold, etc.).


#### **2.2 Simulation Units and Co-Simulation**

A Simulation Unit (SU) packages:
- a dynamical system model (its equations, parameters, internal state), and

- a numerical solver,

behind a black-box interface with well-defined inputs and outputs. Given input trajectories $u_i(t)$, an SU produces output trajectories $y_i(t)$.

A co-simulation is a simulation of a coupled system composed from several SUs $S_i$ that:

- are treated as black boxes (internal details hidden),
- interact only via their input/output variables,
- may themselves be software solvers, real-time controllers, test benches, or physical hardware.

To obtain a global system trajectory, an orchestrator (or co-simulation master) is required to

- manage simulated time,
- schedule the SUs,
- route outputs to inputs according to a co-simulation scenario (connection graph).

The orchestrator + coupled SUs together behave like a single composite SU (a co-SU). This enables hierarchical setups: a co-simulation can itself be used as a building block inside a larger co-simulation.

#### **2.3 Continuous-Time Simulation Units and Communication Grid**

Continuous-time (CT) SUs internally integrate ODE/DAE systems.

Key notions:

- Each SU $S_i$ may uses its own internal step size $h_i$ and its own solver
- The orchestrator defines a communication grid with macro step size $H$

$$ t_n = t_0 + nH, \quad n = 0, 1, 2, \ldots $$

- At these communication times, SUs exchange inputs/outputs.

- Between $t_n$ and $t_{n+1}$ each SU advances independently using internal micro-steps, while seeing its input approximated over the macro step.

- A continuous-time simulation unit can be described as follows:

$$S_i = \langle X_i, U_i, Y_i, \delta_i, \lambda_i, x_i(0), \phi_{U_i} \rangle$$

where
 - $X_i, U_i, Y_i$ are the state, input, and output spaces,
 - $\delta_i$ is the internal state transition over one macro step (including its numerical solver),
 - $\lambda_i$ maps state and inputs to outputs,
 - $x_i(0)$ is the initial state,
 - $\phi_{U_i}$ is the interpolation/extrapolation scheme used to approximate $u_i(t)$ between communication points (e.g. zero-order hold, linear, higher order).


The co-simulation scenario consists of
- a set of SUs $D = \{ S_1, \ldots, S_N \}$,
- their external inputs/outputs,
- a set of coupling equations $L$ that relate outputs to inputs,
- interpolation functions $\phi_{U_{ext}}$ for external inputs.

#### **2.4 FMI, FMUs, and Model Structure**

The Functional Mock-up Interface (FMI) standard provides a tool-independent standard for packaging and exchanging models as Functional Mock-up Units (FMUs). An FMU is a zip file containing:
- a modelDescription.xml (variables, units, causality, variability, dependencies),
- C functions implementing the FMI API for model exchange and co-simulation.
- binaries and additional resources.

FMUs which are exported from OpenModelica are of the FMI version 2.0 and the model description file provides the following information:
- Variable causality: input, output, parameter, calculated parameter, local, etc.
- Variable variability: continuous, discrete, constant, parameter, fixed, tunable
- Variable dependencies in the model structure section:
    - which output depends on which variables
    - which derivative depends on which variables
    - which initial unknonwn depends on which variables

**Model Structure for drive FMU:**
```xml
<ModelStructure>
    <Outputs>
      <!-- Output: torque (index 30) -->
      <Unknown 
        index="30" name="torque"
        dependencies="springDamper.phi_rel, springDamper.w_rel, alpha, omega, phi" 
        causality="(state), (state), input, input, input"
      />
    </Outputs>
</ModelStructure>
```

**Model Structure for pendulum FMU:**
```xml
<ModelStructure>
    <Outputs>
      <!-- Output: alpha (index 6) - angular acceleration -->
      <Unknown 
        index="6" name="alpha"
        dependencies="_D_outputAlias_q, torque" 
        causality="(state), input"
      />
    </Outputs>
</ModelStructure>
```

This structural information is used to
- detect algebraic loops and constraints between FMUs
- identify direct feedthrough (output depending on current input)


### **3. Challenges in Continuous-Time Co-Simulation**

**Problem:** Even if each individual SU is valid and accurate, their composition is not automatically correct

- **Algebraic Constraints:** Physical coupling (e.g. rigid joints, action–reaction forces) yields algebraic equations that must hold across SUs (equal positions/velocities, force balance). With black-box SUs, enforcing these constraints is non-trivial and often requires sensitivities / Jacobians and rollback.

- **Algebraic loops**
    - Couplings can create cycles where variables depend (possibly nonlinearly) on themselves through other SUs.
    - Purely “input–input” loops are already delicate.
    - Loops involving states are more serious and may change the DAE index.
    - Ignoring these loops leads to large errors or instability; they typically require fixed-point or Newton-like iterations over the interface.

- **Accuracy and error control:** Error has several sources:
    - Internal solvers and micro step sizes $h_i$
    - Communication step size $H$
    - Input interpolation $\phi_{U_i}$
    - Reducing $H$ often improves accuracy, but not always; A co-simulation master needs some form of error assessment and possibly step size control.

- **Stability of the coupled system:** Stability is not just a property of each SU. It depends strongly on:
    - coupling topology,
    - orchestration scheme (Jacobi vs Gauss–Seidel vs iterative),
    - interpolation/extrapolation choice.
    - For linear systems, the stability can be studied via the spectral radius of the global error propagation matrix. Iterative schemes (dynamic iteration, IJCSA) often improve stability.

- **Continuity of inputs**
    - CT SUs expect continuous inputs. Piecewise constant extrapolation or sudden changes can:
        - reduce solver efficiency,
        - trigger reinitializations,
        - introduce artificial discontinuities that propagate through the system.


These challenges motivate more sophisticated master algorithms than simple explicit schemes. In the following, we summarize three central orchestration strategies used in this thesis: Jacobi, Gauss–Seidel, and the Interface Jacobian-based Co-Simulation Algorithm (IJCSA).

### **4. Classical Co-SImulation Algorithms**

In this section we consider one communication step from $t_n$ to $t_{n+1}$ with macro step size $H$.

Let
 - $S_i$ be SUs with internal state $x_i$, inputs $u_i$, and outputs $y_i$
 - $L(y, u, u_{ext})$ be a set of coupling equations relating all SU input/ouputs and any external inputs

#### **4.1 Jacobi Co-Simulation Algorithm**

The Jacobi co-simulation algorithm is an **explicit, non-iterative** method where **all SUs are advanced in parallel** using input values from the previous communication point.

1. At communication time $t_n$, each SU has state $x_i^n$ and input $u_i^n$. The **inputs at the next step** $u_i^{n+1}$ are defined by the couplings using only outputs at time $t_n$ (explicit coupling).

2. For the next macro step:
    - The orchestrator freezes each SU's input as some extrapolated function (e.g., zero-order hold or linear).
    - All SUs integrate in **parallel** from $t_n$ to $t_{n+1}$ using these frozen inputs, producing new states $x_i^{n+1}$ and outputs $y_i^{n+1}$.
    $$
    x_i^{n+1} = \delta_i(t_n, x_i^n, u_i^{n}(\cdot)), \quad y_i^{n+1} = \lambda_i(t_{n+1}, x_i^{n+1}, u_i^{n+1})
    $$

3. At $t_n+1$, the new outputs $y_i^{n+1} are available and used to compute the inputs for the next step $u_i^{n+2}$ via the coupling equations.

**Properties:**
- Parallelism: All SUs can run concurrently between $t_n$ and $t_{n+1}$
- Simplicity: No interface iterations or rollback; suitable for loosely coupled systems.
- Phase lag / extrapolation error: Each SU sees “old” information about others; in closed loops this introduces effective delays and can degrade accuracy or stability.

#### **4.2 Gauss-Seidel Co-Simulation Algorithm**

The Gauss-Seidel scheme is sequential and uses the most recent information inside the macro step.

1. Fix an order of SUs, e.g., $S_1, S_2, \ldots  S_N$.

2. At communication time $t_n$:
 - $S_1$ is advanced first from $t_n$ to $t_{n+1}$ with inputs extrapolated from data at $t_n$ or earlier:
 
 $$x_1^{n+1}, y_1^{n+1} = \text{step}(S_1, x_1^n, u_1^{n}(\cdot))$$

3. $S_2$ is advanced next. Its inputs may now use $y_1^{n+1}$, which has just been updated by $S_1$.
Generally:
- Early SUs see *old* outputs from later SUs
- Later SUs see *new* outputs from earlier SUs

4. After all SUs have been advanced, time is increased to $t_{n+1}$.

**Properties**
- Reduced phase lag compared to Jacobi: some couplings use updated outputs within the same macro step.
- Potentially better stability for certain physical orderings (e.g. integrating light masses before heavy masses, or input–output chains in causality order).
- Loss of parallelism: SUs must be advanced sequentially in the chosen order.
- Order sensitivity: The chosen sequence can significantly affect accuracy and stability.

### **5 Interface Jacobian-Based Co-Simulation**

The **Interface Jacobian-based Co-Simulation Algorithm (IJCSA)** (Sicklinger, 2014) generalizes the excplicit schemes by applying a **Newton method at the interface level**, using the Jacobian of the interface constraints.

#### **5.1 Interface Constraint Operator and Residual**

Consider $N$ subsystems(SUs) with input vector $U_i$ and output vector $Y_i$.
Collect all interface inputs and outputs as:

$$U = \begin{bmatrix} U_1 \\ \vdots \\ U_N \end{bmatrix}, \quad
Y = \begin{bmatrix} Y_1 \\ \vdots \\ Y_N \end{bmatrix}$$
​
The couplings are expressed by a possibly nonlinear **interface constraint operator:**

$$L(U, Y) = 0$$

For a simple signall assignment coupling $U_i = Y_j$ for some $i, j$, the constraint operator can be described as:
$$L(U, Y) = U_i - Y_j$$

And the interface residual is defined as 
$$r = L(U, S(U)) = U_i - Y_j$$

where $S(U)$ denotes the composition of all SUs stepped over one macro interval with given interface inputs $U$, producing $Y$.

We are interested in finding interface inputs $U^*$ such that the residual vanishes:
$$r(U^*) = L(U^*, S(U^*)) = 0$$
This ensures that all coupling constraints are satisfied.

#### **5.2 Newton Iteration at the Interface**

The IJCSA applies a Newton method in the space of the interface unknowns $U$ to solve $r(U) = 0$.

1. **Time loop** $t_n \to t_{n+1}$:
Given a current macro step $[t_n, t_{n+1}]$ and a previous solution $U^n$ (or a predictor), initialize the iteration with $U^{(0)} = U^n$

2. **Iteration Loop** for $m = 0, 1, 2, \ldots$:
    1. **Parallel subsystem evaluation:** For all SUs $S_i$ in parallel
        - Integrate internally from $t_n$ to $t_{n+1}$ using current input guess $U_i^{(m)}$ (and appropriate interpolation / extrapolation)
        - Obtain outputs $Y_i^{(m)}$ at $t_{n+1}$
    2. **Residual evaluation**: Evaluate the interface residual
        $$r^{(m)} = L(U^{(m)}, Y^{(m)})$$
        - If $\|r^{(m)}$ is below a tolerance accept $U_{n+1} = U^{(m)}$ and proceed to the next macro step.
    3. **Interface Jacobian Assembly**
        - Compute or approximate the Jacobian
        $$J^{(m)} = \frac{\partial r}{\partial U}\bigg|_{U^{(m)}} = \frac{\partial L}{\partial U} + \frac{\partial L}{\partial Y} \frac{\partial Y}{\partial U}$$
        - This involves contribution form the explicit interface constrain operator $L$, and
        - Local interface Jacobians $\frac{\partial Y_i}{\partial U_j}$ from each $SU$ (from directional derivative if FMU supports or via finite differences)
        - All these blocks are assembled into a global interface Jacobian matrix (dimension is equal to the number of interface unknowns)
    4. **Newton Update**: Solve the linear system
        $$J^{(m)} \Delta U^{(m)} = -r^{(m)}$$
        for the correction $\Delta U^{(m)}$, and optionally apply **relaxation for stability:**
        $$U^{(m+1)} = U^{(m)} + \alpha \Delta U^{(m)}, \quad 0 < \alpha \leq 1$$

3. **Carry over to the next step:**
 - After convergence, the consistent interface inputs $U^{n+1} = U^{(m)}$ can be used as
    - Final inputs for SUs at $t_{n+1}$
    - Initial guess for the next macro step.

**Properties:**
- Implicit coupling: By solving the interface constraints iteratively, IJCSA can handle algebraic loops and tightly coupled systems.
- Parallelism: Each iteration step evaluates all SUs in parallel.
- Jacobian requirement: Accurate Jacobians are crucial for convergence; finite difference approximations can be costly.


**Iteration Sequence for the Newton Method**

$$^{m+1} \mathbf{\phi} = ^{m} \mathbf{\phi} - J(\mathbf{r}(^{m} \mathbf{\phi}))^{-1} \mathbf{r}(^{m} \mathbf{\phi}) $$


$$ J(\mathbf{r}(^{m} \mathbf{\phi}))\underbrace{\left(^{m+1}\mathbf{\phi} - ^{m} \mathbf{\phi}\right)}_{\Delta^{m+1} \mathbf{\phi}} = - \mathbf{r}(^{m} \mathbf{\phi})$$


$$ J(\mathbf{r}(^{m} \mathbf{\phi})) \Delta^{m+1} \mathbf{\phi} = - \mathbf{r}(^{m} \mathbf{\phi})$$

$$ \mathbf{\phi}=\left[\begin{matrix}U_1\\U_2\\\end{matrix}\right],\quad \mathbf{r}=\left[\begin{matrix}R_1\\R_2\\\end{matrix}\right]
$$

where
 - $\mathbf{\phi}$ is the vector of unknowns
 - $\mathbf{r}$ is the residual vector
 - $J$ is the Jacobian operator (dimension only dependent on number of input vraiables at the interface level)
 - $m$ is the iteration index

**Linear Interface System for the Newton Method**

$$
J(r(^{m}\phi)) =
\left[
\begin{matrix}\frac{\partial R_1}{\partial U_1}&\frac{\partial R_1}{\partial U_2}\\
\frac{\partial R_2}{\partial U_1}&\frac{\partial R_2}{\partial U_2}\\
\end{matrix}
\right]
=
\left[
\begin{matrix}\frac{\partial I_1}{\partial U_1} & \frac{\partial I_1}{\partial U_2}\\
\frac{\partial I_2}{\partial U_1} & \frac{\partial I_2}{\partial U_2}\\
\end{matrix}
\right]
=
\left[
\begin{matrix}\frac{\partial (U_1 - Y_2)}{\partial U_1}&\frac{\partial (U_1 - Y_2)}{\partial U_2}\\
\frac{\partial (U_2 - Y_1)}{\partial U_1} & \frac{\partial (U_2 - Y_1)}{\partial U_2}\\
\end{matrix}
\right]
=
\left[
\begin{matrix}\ I & -\frac{\partial (Y_2)}{\partial U_2}\\
-\frac{\partial (Y_1)}{\partial U_1} & I\\
\end{matrix}
\right]
$$

$$
\left[
\begin{matrix}\ I & -\frac{\partial (Y_2)}{\partial U_2}\\
-\frac{\partial (Y_1)}{\partial U_1} & I\\
\end{matrix}
\right] 

\left[
\begin{matrix} \Delta U_1 \\ \Delta U_2 \end{matrix}
\right] 

= 

- \left[
\begin{matrix} \Delta R_1 \\ \Delta R_2 \end{matrix}
\right]

$$

#### **5.3 Practical Considerations for FMU-based Co-Simulation**

**Availability of directional derivatives**
- If CS-FMUs provide fmi2GetDirectionalDerivative, local interface Jacobians $\frac{\partial Y_i}{\partial U_j}$ can be evaluated efficiently
- Otherwise, finite difference approximations can be used at the cost of extra FMU evaluations.

**FMU state management**
- A conceptually clean interface Newton step assumes one can:
    - Save the FMU state at $t_n$,
    - perform trial macro steps with different inputs,
    - rollback to the saved state when needed.

- **FMI 2.0**
    - offers getFMUstate/setFMUstate, but many tools (such as your OpenModelica-generated CS-FMUs) do not support them.
    - In such cases, one must emulate rollback by re-initializing the FMU and reapplying consistent start conditions, or design the iteration so that it only uses local output evaluations at fixed time with 
    - $dt = 0$(no state advancement), combined with careful handling of direct feedthrough.

**Use of ModelStructure**
 - Detect which outputs have direct feedthrough from which inputs,
 - Identify algebraic loops (cycles where outputs depend directly on current inputs),
 - Focus the Jacobian on the subset of variables that actually participate in interface constraints.